## Phase 1: Data Extraction from Steam Store

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://store.steampowered.com/search/?filter=globaltopsellers"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
}

try: 
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        print("✅ Access granted! Connected to Steam successfully.")
    else: 
        print(f"❌ Connection error: {response.status_code}")
except Exception as e:
    print(f"⚠️ An unexpected error occurred: {e}")
        

✅ Access granted! Connected to Steam successfully.


In [2]:
soup = BeautifulSoup(response.text, 'html.parser')
game_listings = soup.find_all('a', class_='search_result_row')

print(f"📦 Successfully identified {len(game_listings)} games on this page.")

if game_listings:
    first_game_title = game_listings[0].find('span', class_='title').text
    print(f"🎮 The first title in the collection is: {first_game_title}")

📦 Successfully identified 50 games on this page.
🎮 The first title in the collection is: Counter-Strike 2


In [25]:
game_data = []

for game in game_listings:
    title = game.find('span', class_='title').text if game.find('span', class_='title') else "N/A"
    
    price_element = game.find('div', class_='discount_final_price')
    
    if price_element:
        price = price_element.get_text(strip=True)
    else:
        fallback_price = game.find('div', class_='search_price')
        price = fallback_price.get_text(strip=True) if fallback_price else "TBA"

    try:
        review_element = game.find('span', class_='search_review_summary')
        if review_element and 'data-tooltip-html' in review_element.attrs:
            raw_review = review_element['data-tooltip-html']
            clean_review = raw_review.replace('<br>', ' ').replace('&nbsp;', ' ').strip()
        else:
            clean_review = "No Reviews"
    except Exception:
        clean_review = "Error Parsing Reviews"

    game_data.append({
        'Title': title,
        'Price': price,
        'Review_Summary': clean_review
    })

df = pd.DataFrame(game_data)
print(f"✅ Success! Managed to extract {len(df)} titles with their respective prices.")
df.head(10)

✅ Success! Managed to extract 50 titles with their respective prices.


,Title,Price,Review_Summary
0,Counter-Strike 2,Free,"Very Positive 86% of the 2,508,216 user review..."
1,Crimson Desert,"Mex$ 1,299.00","Very Positive 83% of the 29,891 user reviews f..."
2,Apex Legends™,Free,"Mostly Positive 76% of the 438,048 user review..."
3,Marvel Rivals,Free,"Mostly Positive 78% of the 273,104 user review..."
4,Slay the Spire 2,Mex$ 290.00,"Overwhelmingly Positive 95% of the 35,636 user..."
5,Warframe,Free,"Very Positive 91% of the 289,295 user reviews ..."
6,Steam Deck,TBA,No Reviews
7,Gray Zone Warfare,Mex$ 381.83,"Mostly Positive 71% of the 40,526 user reviews..."
8,PUBG: BATTLEGROUNDS,Free,"Mixed 66% of the 443,263 user reviews for this..."
9,ARC Raiders,Mex$ 607.20,"Very Positive 86% of the 180,543 user reviews ..."


In [32]:
output_path = '../data/raw_steam_data.csv'

df.to_csv(output_path, index=False)
print(f"💾 Data successfully persisted at: {output_path}")

💾 Data successfully persisted at: ../data/raw_steam_data.csv
